<a href="https://colab.research.google.com/github/KiranDhanvate/ai-job-aggregator/blob/main/convdeepfm_best_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import math
from collections import defaultdict

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import roc_auc_score


In [3]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [4]:
DATA_PATH = "/content/drive/MyDrive/data/convdeepfm_ranking_standard_dataset.csv"  # Placeholder
df = pd.read_csv(DATA_PATH)



In [5]:
df

,user_id,job_id,session_id,timestamp,user_text,job_text,user_skills,job_skills,user_education,job_type,location,user_experience_years,job_required_experience,experience_gap,label
0,U0401,J0241,1,03-06-2024 16.19,Machine Learning Engineer with strong Deep Lea...,Exciting opportunity for React Developer in IT...,"Swift, Financial Modeling, React, Client Manag...","Swift, Flutter, Agile, GraphQL",B.Sc in Computer Science,Full-time,"Kolkata, West Bengal",8,9,-1,0.0
1,U0840,J0181,1,07-04-2024 9.36,Machine Learning Engineer from B.Tech in Compu...,Product Manager position at fast-growing BFSI ...,"Cloud Computing, AWS, SEO, API Development, Re...","UI/UX Design, Kubernetes, Deep Learning, Sprin...",B.Tech in Computer Science,Full-time,"Kochi, Kerala",7,1,6,1.0
2,U0170,J0263,5,27-05-2024 5.54,Professional HR Manager looking for challengin...,Immediate opening for Full Stack Web Developer...,"Attention to Detail, Spark, Creativity, Market...","Fintech, JavaScript, React, REST API, DevOps, ...",B.Tech in Computer Science,Contract,"Coimbatore, Tamil Nadu",10,1,9,0.0
3,U0721,J0045,4,09-02-2024 0.02,Skilled DevOps Engineer with expertise in Flut...,Join our growing team as Talent Acquisition Sp...,"Kafka, Team Collaboration, Angular, CSS, Machi...","E-commerce, CSS, Time Management, Redis",MCA (Master of Computer Applications),Hybrid,"Coimbatore, Tamil Nadu",9,0,9,0.0
4,U0333,J0082,5,30-05-2024 6.53,7 years as Data Scientist. Specialized in Cybe...,Talent Acquisition Specialist position at fast...,"Spark, Docker, GCP, Financial Modeling, Azure,...","Java, Scikit-learn, C++, Business Intelligence...",B.Des in Design,Full-time,"Gurgaon, Haryana",3,9,-6,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11761,U0960,J0272,4,08-02-2024 8.44,Professional Scrum Master looking for challeng...,Hiring Senior Financial Analyst with strong El...,"Fintech, Attention to Detail, Financial Modeli...","Kubernetes, React, Flask",B.Tech in Computer Science,Contract,"Indore, Madhya Pradesh",10,10,0,0.0
11762,U0443,J0198,2,14-02-2024 1.24,Motivated Software Engineer with 7+ years expe...,Hiring Android Developer with strong Git skill...,"Market Research, Adaptability, React, Elastics...","React, Creativity, Redis",B.Tech in Computer Science,Internship,"Mumbai, Maharashtra",3,4,-1,0.0
11763,U0460,J0452,3,27-05-2024 16.46,Project Manager from MCA (Master of Computer A...,We are looking for Full Stack Web Developer to...,"Swift, Spark, PostgreSQL, SEO, SEM, Deep Learn...","HTML, PyTorch, Hadoop, Content Strategy, SEM, ...",MCA (Master of Computer Applications),Work from Home,"Coimbatore, Tamil Nadu",2,9,-7,0.0
11764,U0074,J0326,1,16-02-2024 22.48,Experienced Content Writer with 4 years in Scr...,We are looking for Cloud Solutions Architect t...,"E-commerce, Communication, Kotlin, Creativity,...","Docker, Scrum, GraphQL, Redis, Adaptability, F...",B.Tech in Electronics and Communication,Work from Home,"Hyderabad, Telangana",5,7,-2,0.0


In [6]:
CAT_COLS = ["user_id", "job_id", "user_education", "job_type", "location"]
field_dims = []

for col in CAT_COLS:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    field_dims.append(df[col].nunique())


In [7]:
df = df.sort_values("timestamp").reset_index(drop=True)

n = len(df)
train_df = df.iloc[:int(0.7 * n)]
val_df   = df.iloc[int(0.7 * n):int(0.85 * n)]
test_df  = df.iloc[int(0.85 * n):]


In [8]:
tfidf = TfidfVectorizer(max_features=300)
tfidf.fit(df["user_text"].fillna("") + " " + df["job_text"].fillna(""))

def text_features(frame):
    u = tfidf.transform(frame["user_text"].fillna(""))
    j = tfidf.transform(frame["job_text"].fillna(""))
    return np.hstack([u.toarray(), j.toarray()])

NUM_COLS = ["user_experience_years", "job_required_experience", "experience_gap"]

def numeric_features(frame):
    return frame[NUM_COLS].values.astype(np.float32)


In [9]:
class BPRJobDataset(Dataset):
    def __init__(self, frame):
        self.samples = []

        for user, group in frame.groupby("user_id"):
            pos = group[group["label"] > 0]
            neg = group[group["label"] == 0]

            if len(pos) == 0 or len(neg) == 0:
                continue

            for _, p in pos.iterrows():
                hard_neg = neg[
                    (neg["job_type"] == p["job_type"]) |
                    (neg["location"] == p["location"])
                ]

                n = hard_neg.sample(1).iloc[0] if len(hard_neg) > 0 else neg.sample(1).iloc[0]
                self.samples.append((p, n))

        print(f"[BPRDataset] samples: {len(self.samples)}")

    def __len__(self):
        return len(self.samples)

    def build_x(self, row):
        x_cat = row[CAT_COLS].values.astype(np.int64)
        x_deep = np.hstack([
            text_features(pd.DataFrame([row])),
            numeric_features(pd.DataFrame([row]))
        ]).squeeze(0)

        return torch.LongTensor(x_cat), torch.FloatTensor(x_deep)

    def __getitem__(self, idx):
        pos, neg = self.samples[idx]
        return self.build_x(pos), self.build_x(neg)


In [10]:
class ConvDeepFM(nn.Module):
    def __init__(self, field_dims, deep_dim, embed_dim=32):
        super().__init__()

        self.embedding = nn.Embedding(sum(field_dims), embed_dim)
        self.offsets = torch.tensor(
            np.array((0, *np.cumsum(field_dims)[:-1])), dtype=torch.long
        )

        self.embed_dropout = nn.Dropout(0.2)

        self.convs = nn.ModuleList([
            nn.Conv1d(embed_dim, 64, k) for k in [2, 3, 4]
        ])

        self.deep = nn.Sequential(
            nn.Linear(64 * 3 + deep_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(128, 1)
        )

        self.fusion = nn.Linear(2, 1)

    def forward(self, x_cat, x_deep):
        x_cat = x_cat + self.offsets.to(x_cat.device)
        emb = self.embed_dropout(self.embedding(x_cat))

        # FM
        sum_sq = torch.sum(emb, dim=1) ** 2
        sq_sum = torch.sum(emb ** 2, dim=1)
        fm_out = 0.5 * torch.sum(sum_sq - sq_sum, dim=1, keepdim=True)

        # CNN
        x = emb.permute(0, 2, 1)
        cnn_out = torch.cat(
            [torch.max(torch.relu(conv(x)), dim=2)[0] for conv in self.convs],
            dim=1
        )

        deep_out = self.deep(torch.cat([cnn_out, x_deep], dim=1))
        return self.fusion(torch.cat([fm_out, deep_out], dim=1)).squeeze(1)


In [11]:
df

,user_id,job_id,session_id,timestamp,user_text,job_text,user_skills,job_skills,user_education,job_type,location,user_experience_years,job_required_experience,experience_gap,label
0,264,210,3,01-01-2024 10.55,Motivated Backend Developer with 6+ years expe...,We are looking for AI/ML Engineer to join our ...,"Vue.js, GCP, TypeScript, Deep Learning","TypeScript, Problem Solving, HTML, Adaptabilit...",3,2,7,7,6,1,0.0
1,264,62,3,01-01-2024 10.55,Motivated Backend Developer with 6+ years expe...,We are looking for AI/ML Engineer to join our ...,"Vue.js, GCP, TypeScript, Deep Learning","TypeScript, Problem Solving, HTML, Adaptabilit...",3,2,7,7,6,1,0.0
2,264,226,3,01-01-2024 10.55,Motivated Backend Developer with 6+ years expe...,We are looking for AI/ML Engineer to join our ...,"Vue.js, GCP, TypeScript, Deep Learning","TypeScript, Problem Solving, HTML, Adaptabilit...",3,2,7,7,6,1,0.0
3,264,122,3,01-01-2024 10.55,Motivated Backend Developer with 6+ years expe...,We are looking for AI/ML Engineer to join our ...,"Vue.js, GCP, TypeScript, Deep Learning","TypeScript, Problem Solving, HTML, Adaptabilit...",3,2,7,7,6,1,0.0
4,264,390,3,01-01-2024 10.55,Motivated Backend Developer with 6+ years expe...,We are looking for AI/ML Engineer to join our ...,"Vue.js, GCP, TypeScript, Deep Learning","TypeScript, Problem Solving, HTML, Adaptabilit...",3,2,7,7,6,1,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11761,243,187,4,31-05-2024 9.52,2 years as Cybersecurity Analyst. Specialized ...,Exciting opportunity for Full Stack Web Develo...,"Cloud Computing, Deep Learning, JavaScript, Gi...","Agile, GCP, HTML, Attention to Detail, Present...",10,3,11,8,10,-2,0.0
11762,243,172,4,31-05-2024 9.52,2 years as Cybersecurity Analyst. Specialized ...,Exciting opportunity for Full Stack Web Develo...,"Cloud Computing, Deep Learning, JavaScript, Gi...","Agile, GCP, HTML, Attention to Detail, Present...",10,3,11,8,10,-2,0.0
11763,243,430,4,31-05-2024 9.52,2 years as Cybersecurity Analyst. Specialized ...,Exciting opportunity for Full Stack Web Develo...,"Cloud Computing, Deep Learning, JavaScript, Gi...","Agile, GCP, HTML, Attention to Detail, Present...",10,3,11,8,10,-2,0.0
11764,243,259,4,31-05-2024 9.52,2 years as Cybersecurity Analyst. Specialized ...,Exciting opportunity for Full Stack Web Develo...,"Cloud Computing, Deep Learning, JavaScript, Gi...","Agile, GCP, HTML, Attention to Detail, Present...",10,3,11,8,10,-2,0.0


In [12]:
bce_loss_fn = nn.BCEWithLogitsLoss()

def hybrid_loss(pos_score, neg_score, lambda_bce=0.5):
    bpr = -torch.mean(torch.log(torch.sigmoid(pos_score - neg_score) + 1e-8))
    labels = torch.cat([torch.ones_like(pos_score), torch.zeros_like(neg_score)])
    preds = torch.cat([pos_score, neg_score])
    bce = bce_loss_fn(preds, labels)
    return bpr + lambda_bce * bce


In [13]:
device = "cuda" if torch.cuda.is_available() else "cpu"

train_ds = BPRJobDataset(train_df)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)

sample_deep_dim = train_ds[0][0][1].shape[0]

model = ConvDeepFM(field_dims, sample_deep_dim).to(device)

optimizer = optim.Adam(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-5
)

best_ndcg = 0
patience = 3
patience_counter = 0

for epoch in range(20):
    model.train()
    total_loss = 0

    for (xc_p, xd_p), (xc_n, xd_n) in train_loader:
        xc_p, xd_p = xc_p.to(device), xd_p.to(device)
        xc_n, xd_n = xc_n.to(device), xd_n.to(device)

        optimizer.zero_grad()
        loss = hybrid_loss(model(xc_p, xd_p), model(xc_n, xd_n))
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1} | Loss: {total_loss/len(train_loader):.4f}")


[BPRDataset] samples: 1372
Epoch 1 | Loss: 6.2577
Epoch 2 | Loss: 6.0258
Epoch 3 | Loss: 5.5371
Epoch 4 | Loss: 5.2767
Epoch 5 | Loss: 4.6970
Epoch 6 | Loss: 4.4770
Epoch 7 | Loss: 3.8910
Epoch 8 | Loss: 3.4759
Epoch 9 | Loss: 3.0643
Epoch 10 | Loss: 2.6319
Epoch 11 | Loss: 2.2586
Epoch 12 | Loss: 2.0667
Epoch 13 | Loss: 1.6158
Epoch 14 | Loss: 1.4125
Epoch 15 | Loss: 1.2012
Epoch 16 | Loss: 1.0533
Epoch 17 | Loss: 0.9997
Epoch 18 | Loss: 0.8806
Epoch 19 | Loss: 0.8453
Epoch 20 | Loss: 0.7215


In [14]:
def precision_at_k(rankings, k):
    return np.mean([
        sum(l for _, l in sorted(v, reverse=True)[:k]) / k
        for v in rankings.values() if sum(l for _, l in v) > 0
    ])

def recall_at_k(rankings, k):
    return np.mean([
        sum(l for _, l in sorted(v, reverse=True)[:k]) / sum(l for _, l in v)
        for v in rankings.values() if sum(l for _, l in v) > 0
    ])

def ndcg_at_k(rankings, k):
    scores = []
    for v in rankings.values():
        if sum(l for _, l in v) == 0:
            continue
        ranked = sorted(v, key=lambda x: x[0], reverse=True)
        dcg = sum((2**l - 1) / math.log2(i + 2) for i, (_, l) in enumerate(ranked[:k]))
        ideal = sorted(v, key=lambda x: x[1], reverse=True)
        idcg = sum((2**l - 1) / math.log2(i + 2) for i, (_, l) in enumerate(ideal[:k]))
        scores.append(dcg / idcg)
    return np.mean(scores)

def hit_at_k(rankings, k):
    return np.mean([
        1 if any(l == 1 for _, l in sorted(v, reverse=True)[:k]) else 0
        for v in rankings.values()
    ])
def average_precision(ranked, k):
    score = 0.0
    hits = 0
    for i, (_, label) in enumerate(ranked[:k]):
        if label == 1:
            hits += 1
            score += hits / (i + 1)
    return score / max(1, hits)

def map_at_k(rankings, k):
    aps = []
    for v in rankings.values():
        if sum(l for _, l in v) == 0:
            continue
        ranked = sorted(v, key=lambda x: x[0], reverse=True)
        aps.append(average_precision(ranked, k))
    return sum(aps) / len(aps)
def mrr(rankings):
    rr = []
    for v in rankings.values():
        ranked = sorted(v, key=lambda x: x[0], reverse=True)
        for i, (_, label) in enumerate(ranked):
            if label == 1:
                rr.append(1 / (i + 1))
                break
    return sum(rr) / len(rr)
def coverage_at_k(rankings, k):
    recommended_jobs = set()
    for v in rankings.values():
        ranked = sorted(v, key=lambda x: x[0], reverse=True)[:k]
        for job in ranked:
            recommended_jobs.add(job)
    return len(recommended_jobs)



In [16]:
from collections import defaultdict

model.eval()

user_rankings = defaultdict(list)
labels_all = []
scores_all = []

with torch.no_grad():
    for _, row in test_df.iterrows():
        # categorical features
        x_cat = torch.LongTensor(
            row[CAT_COLS].values.astype(np.int64)
        ).unsqueeze(0).to(device)

        # deep features
        x_deep = torch.FloatTensor(
            np.hstack([
                text_features(pd.DataFrame([row])),
                numeric_features(pd.DataFrame([row]))
            ])
        ).to(device)

        score = model(x_cat, x_deep).item()
        label = int(row["label"])

        user_rankings[row["user_id"]].append((score, label))
        scores_all.append(score)
        labels_all.append(label)


In [19]:
K = 10

print(f"Precision@{K}: {precision_at_k(user_rankings, K):.4f}")
print(f"Recall@{K}:    {recall_at_k(user_rankings, K):.4f}")
print(f"NDCG@{K}:      {ndcg_at_k(user_rankings, K):.4f}")
print(f"Hit@{K}:       {hit_at_k(user_rankings, K):.4f}")
print(f"MAP@{K}:       {map_at_k(user_rankings, K):.4f}")
print(f"MRR:           {mrr(user_rankings):.4f}")



Precision@10: 0.1760
Recall@10:    0.8622
NDCG@10:      0.4523
Hit@10:       0.9007
MAP@10:       0.3215
MRR:           0.2908


In [20]:
labels_auc = [1 if l > 0 else 0 for l in labels_all]


In [21]:
from sklearn.metrics import roc_auc_score

print("AUC:", roc_auc_score(labels_auc, scores_all))


AUC: 0.5065437459824175


In [22]:
torch.save(model.state_dict(), "convdeepfm_best.pth")


In [23]:
model = ConvDeepFM(
    field_dims=field_dims,
    deep_dim=sample_deep_dim
).to(device)

model.load_state_dict(torch.load("convdeepfm_best.pth", map_location=device))
model.eval()


ConvDeepFM(
  (embedding): Embedding(1481, 32)
  (embed_dropout): Dropout(p=0.2, inplace=False)
  (convs): ModuleList(
    (0): Conv1d(32, 64, kernel_size=(2,), stride=(1,))
    (1): Conv1d(32, 64, kernel_size=(3,), stride=(1,))
    (2): Conv1d(32, 64, kernel_size=(4,), stride=(1,))
  )
  (deep): Sequential(
    (0): Linear(in_features=651, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.4, inplace=False)
    (3): Linear(in_features=128, out_features=1, bias=True)
  )
  (fusion): Linear(in_features=2, out_features=1, bias=True)
)

In [24]:
torch.save({
    "epoch": epoch,
    "model_state": model.state_dict(),
    "optimizer_state": optimizer.state_dict(),
    "best_ndcg": best_ndcg
}, "convdeepfm_checkpoint.pth")


In [25]:
checkpoint = torch.load("convdeepfm_checkpoint.pth", map_location=device)

model.load_state_dict(checkpoint["model_state"])
optimizer.load_state_dict(checkpoint["optimizer_state"])

start_epoch = checkpoint["epoch"] + 1
best_ndcg = checkpoint["best_ndcg"]


In [26]:
metadata = {
    "field_dims": field_dims,
    "deep_dim": sample_deep_dim,
    "cat_cols": CAT_COLS,
    "num_cols": NUM_COLS,
    "tfidf_dim": tfidf.max_features,
}

torch.save(metadata, "convdeepfm_meta.pth")


In [27]:
with torch.no_grad():
    score = model(x_cat, x_deep)
    print(score)


tensor([-1.8825])
